In [1]:
import boto3
import sagemaker
import pandas as pd
import json
from sagemaker import get_execution_role
from sagemaker.model import Model
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DataCaptureConfig,
    CronExpressionGenerator
)
from sagemaker.model_monitor.dataset_format import DatasetFormat

# Setup
session = sagemaker.Session()
role = get_execution_role()  # or paste your IAM role ARN directly
region = boto3.Session().region_name
bucket = "placement-project-bkt"

print(f"Role: {role}")
print(f"Region: {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::218736973000:role/service-role/AmazonSageMaker-ExecutionRole-20260209T120967
Region: eu-north-1


In [2]:
from sagemaker.model_monitor import ModelQualityMonitor

model_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

In [29]:
# import boto3, time

# sm_client = boto3.client("sagemaker")

# # Delete both schedules
# for schedule_name in ["Drift-Monitor", "Model-Quality-Monitor"]:
#     try:
#         sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
#         print(f"🗑️ {schedule_name} deleted!")
#     except Exception as e:
#         print(f"No existing schedule {schedule_name}: {e}")

# time.sleep(10)
# print("✅ All clear!")

🗑️ Drift-Monitor deleted!
🗑️ Model-Quality-Monitor deleted!
✅ All clear!


In [15]:
import pandas as pd

baseline_df = pd.DataFrame({
    "probability": [0.98, 0.19, 0.97, 0.67, 0.08, 0.95, 0.32, 0.88],
    "prediction":  [1,    0,    1,    1,    0,    1,    0,    1   ],
    "label":       [1,    0,    1,    0,    0,    1,    1,    1   ]
})

baseline_df.to_csv("model_quality_baseline.csv", index=False)
print("✅ Baseline CSV created!")
print(baseline_df)

✅ Baseline CSV created!
   probability  prediction  label
0         0.98           1      1
1         0.19           0      0
2         0.97           1      1
3         0.67           1      0
4         0.08           0      0
5         0.95           1      1
6         0.32           0      1
7         0.88           1      1


In [16]:
import sagemaker

baseline_s3_uri = sagemaker.Session().upload_data(
    path="model_quality_baseline.csv",
    bucket="placement-project-bkt",
    key_prefix="monitoring/model-quality-baseline"
)
print(f"✅ Baseline uploaded to: {baseline_s3_uri}")

✅ Baseline uploaded to: s3://placement-project-bkt/monitoring/model-quality-baseline/model_quality_baseline.csv


In [32]:
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

role = sagemaker.get_execution_role()

model_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
)

model_monitor.suggest_baseline(
    baseline_dataset=baseline_s3_uri,
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
    probability_threshold_attribute="0.5",
    output_s3_uri="s3://placement-project-bkt/monitoring/model-quality-baseline-output/"
)

model_monitor.latest_baselining_job.wait(logs=False)
print("✅ Baseline job completed!")
print(f"📊 Accuracy : {model_monitor.baseline_statistics().body_dict['binary_classification_metrics']['accuracy']['value']}")
print(f"📊 F1       : {model_monitor.baseline_statistics().body_dict['binary_classification_metrics']['f1']['value']}")
print(f"📊 AUC      : {model_monitor.baseline_statistics().body_dict['binary_classification_metrics']['auc']['value']}")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-04-06-07-28-58-792


...........................................................!✅ Baseline job completed!
📊 Accuracy : 0.75
📊 F1       : 0.8000000000000002
📊 AUC      : 0.9333333333333333


In [3]:
import boto3, time, sagemaker
from sagemaker.model_monitor import ModelQualityMonitor, EndpointInput, CronExpressionGenerator

# Delete old schedule
sm_client = boto3.client("sagemaker")
try:
    sm_client.delete_monitoring_schedule(MonitoringScheduleName="Model-Quality-Monitor")
    print("🗑️ Old schedule deleted!")
except Exception as e:
    print(f"No existing schedule: {e}")

time.sleep(10)

role = sagemaker.get_execution_role()
model_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
)

model_monitor.create_monitoring_schedule(
    monitor_schedule_name="Model-Quality-Monitor",
    endpoint_input=EndpointInput(
        endpoint_name="Driftplacement-model-endpointt",
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",
        probability_attribute="0",
        probability_threshold_attribute=0.5,
    ),
    ground_truth_input="s3://placement-project-bkt/monitoring/ground-truth",  # ✅ no trailing slash
    problem_type="BinaryClassification",
    output_s3_uri="s3://placement-project-bkt/monitoring/model-quality-reports/",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

status = model_monitor.describe_schedule()
print(f"✅ Schedule recreated! Status: {status['MonitoringScheduleStatus']}")

🗑️ Old schedule deleted!
✅ Schedule recreated! Status: Pending


In [4]:
import time
from sagemaker.model_monitor import ModelQualityMonitor

model_monitor = ModelQualityMonitor.attach("Model-Quality-Monitor")

while True:
    status = model_monitor.describe_schedule()
    current_status = status["MonitoringScheduleStatus"]
    print(f"📋 Status: {current_status}")
    
    if current_status == "Scheduled":
        print("✅ Schedule is active!")
        break
    
    time.sleep(10)

📋 Status: Scheduled
✅ Schedule is active!


In [17]:
from datetime import datetime, timezone, timedelta

status = model_monitor.describe_schedule()

print(f"📋 Schedule Status : {status['MonitoringScheduleStatus']}")
print(f"⏰ Schedule Expr   : {status['MonitoringScheduleConfig']['ScheduleConfig']['ScheduleExpression']}")

# Current times
utc_now = datetime.now(timezone.utc)
ist_now = utc_now + timedelta(hours=5, minutes=30)

# Next run = next top of the hour
next_utc = utc_now.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)
next_ist = next_utc + timedelta(hours=5, minutes=30)

print(f"\n🕐 Current IST Time : {ist_now.strftime('%H:%M:%S')}")
print(f"⏭️  Next Run IST     : {next_ist.strftime('%H:%M:%S')} today")
minutes_left = (next_utc - utc_now).seconds // 60
print(f"⏳ Time remaining   : {minutes_left} minutes")

📋 Schedule Status : Scheduled
⏰ Schedule Expr   : cron(0 * ? * * *)

🕐 Current IST Time : 15:19:25
⏭️  Next Run IST     : 15:30:00 today
⏳ Time remaining   : 10 minutes


In [8]:
import boto3

runtime = boto3.client("sagemaker-runtime")

test_inputs = [
    "7.5,105,72",
    "6.2,98,55",
    "8.1,112,85",
    "5.5,90,40",
    "9.0,118,95",
]

event_ids = []
for i, data in enumerate(test_inputs):
    response = runtime.invoke_endpoint(
        EndpointName="Driftplacement-model-endpointt",
        ContentType="text/csv",
        Body=data
    )
    result = response["Body"].read().decode("utf-8").strip()
    print(f"✅ Request {i+1} | Input: {data} | Probability: {result}")

✅ Request 1 | Input: 7.5,105,72 | Probability: 0.9795895218849182
✅ Request 2 | Input: 6.2,98,55 | Probability: 0.9829667806625366
✅ Request 3 | Input: 8.1,112,85 | Probability: 0.05536679923534393
✅ Request 4 | Input: 5.5,90,40 | Probability: 0.7584520578384399
✅ Request 5 | Input: 9.0,118,95 | Probability: 0.19305570423603058


In [9]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="placement-project-bkt",
    Prefix="monitoring/data-capture/"
)

files = response.get("Contents", [])

if not files:
    print("❌ No captured files found!")
else:
    print(f"✅ Found {len(files)} files:\n")
    for obj in sorted(files, key=lambda x: x["LastModified"], reverse=True):
        print(f"🕐 {obj['LastModified']}  |  {obj['Key']}")

✅ Found 1 files:

🕐 2026-04-06 07:41:04+00:00  |  monitoring/data-capture/Driftplacement-model-endpointt/AllTraffic/2026/04/06/07/39-03-827-fbee0697-ef7e-4d4c-aacb-69c2290338ff.jsonl


In [10]:
import boto3, json

s3 = boto3.client("s3")

# ✅ Exact path from above
obj = s3.get_object(
    Bucket="placement-project-bkt",
    Key="monitoring/data-capture/Driftplacement-model-endpointt/AllTraffic/2026/04/06/07/39-03-827-fbee0697-ef7e-4d4c-aacb-69c2290338ff.jsonl"
)
content = obj["Body"].read().decode("utf-8")

events = []
for line in content.strip().split("\n"):
    record = json.loads(line)
    event_id   = record["eventMetadata"]["eventId"]
    input_data = record["captureData"]["endpointInput"]["data"]
    prediction = record["captureData"]["endpointOutput"]["data"].strip()
    pred_label = 1 if float(prediction) >= 0.5 else 0
    
    events.append({
        "eventId": event_id,
        "input": input_data,
        "prediction": prediction
    })
    
    print(f"🔑 EventId    : {event_id}")
    print(f"📥 Input      : {input_data}")
    print(f"📤 Probability: {prediction}")
    print(f"🎯 Pred Label : {pred_label}")
    print("-" * 50)

print(f"\n✅ Total events captured: {len(events)}")

🔑 EventId    : ebafd9a7-8557-4474-965d-c317886564f7
📥 Input      : 7.5,105,72
📤 Probability: 0.9795895218849182
🎯 Pred Label : 1
--------------------------------------------------
🔑 EventId    : 1b191eba-89ae-49df-9369-a450678e799b
📥 Input      : 6.2,98,55
📤 Probability: 0.9829667806625366
🎯 Pred Label : 1
--------------------------------------------------
🔑 EventId    : 825996cc-f22d-47cb-a68f-ba6e93461093
📥 Input      : 8.1,112,85
📤 Probability: 0.05536679923534393
🎯 Pred Label : 0
--------------------------------------------------
🔑 EventId    : b1efb257-bcba-4e89-9b88-419777020aeb
📥 Input      : 5.5,90,40
📤 Probability: 0.7584520578384399
🎯 Pred Label : 1
--------------------------------------------------
🔑 EventId    : 3b8187a0-d51a-4c22-ade2-6c2708935fa9
📥 Input      : 9.0,118,95
📤 Probability: 0.19305570423603058
🎯 Pred Label : 0
--------------------------------------------------
🔑 EventId    : d7d9ceb9-fb55-4903-81a6-1fd301aeb7a9
📥 Input      : 7.5,105,72
📤 Probability: 0.97958

In [11]:
import boto3, json
from datetime import datetime

s3 = boto3.client("s3")

# ✅ Real actual placement results for each student
# Match with predictions:
# 1: 7.5,105,72  → prob 0.98 → pred 1 → actual 1 (got placed)
# 2: 6.2,98,55   → prob 0.98 → pred 1 → actual 0 (did not get placed)
# 3: 8.1,112,85  → prob 0.05 → pred 0 → actual 0 (did not get placed)
# 4: 5.5,90,40   → prob 0.75 → pred 1 → actual 0 (did not get placed)
# 5: 9.0,118,95  → prob 0.19 → pred 0 → actual 0 (did not get placed)
# 6: 7.5,105,72  → prob 0.98 → pred 1 → actual 1 (got placed)
# 7: 6.2,98,55   → prob 0.98 → pred 1 → actual 0 (did not get placed)
# 8: 8.1,112,85  → prob 0.05 → pred 0 → actual 1 (got placed)
# 9: 5.5,90,40   → prob 0.75 → pred 1 → actual 0 (did not get placed)
# 10: 9.0,118,95 → prob 0.19 → pred 0 → actual 0 (did not get placed)

actual_labels = [1, 0, 0, 0, 0, 1, 0, 1, 0, 0]  #  simulated actual results

ground_truths = []
for i, event in enumerate(events):
    ground_truths.append({
        "groundTruthData": {
            "data": str(actual_labels[i]),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": event["eventId"],  #  real event ID
            "inferenceTime": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        "eventVersion": "0"
    })
    print(f" Student {i+1} | Input: {event['input']} | Prob: {event['prediction']} | Actual: {actual_labels[i]}")

# Upload to S3
now = datetime.utcnow()
key = f"monitoring/ground-truth/{now.year}/{now.month:02d}/{now.day:02d}/{now.hour:02d}/labels.jsonl"

s3.put_object(
    Bucket="placement-project-bkt",
    Key=key,
    Body="\n".join([json.dumps(gt) for gt in ground_truths])
)
print(f"\n Ground truth uploaded to: s3://placement-project-bkt/{key}")

 Student 1 | Input: 7.5,105,72 | Prob: 0.9795895218849182 | Actual: 1
 Student 2 | Input: 6.2,98,55 | Prob: 0.9829667806625366 | Actual: 0
 Student 3 | Input: 8.1,112,85 | Prob: 0.05536679923534393 | Actual: 0
 Student 4 | Input: 5.5,90,40 | Prob: 0.7584520578384399 | Actual: 0
 Student 5 | Input: 9.0,118,95 | Prob: 0.19305570423603058 | Actual: 0
 Student 6 | Input: 7.5,105,72 | Prob: 0.9795895218849182 | Actual: 1
 Student 7 | Input: 6.2,98,55 | Prob: 0.9829667806625366 | Actual: 0
 Student 8 | Input: 8.1,112,85 | Prob: 0.05536679923534393 | Actual: 1
 Student 9 | Input: 5.5,90,40 | Prob: 0.7584520578384399 | Actual: 0
 Student 10 | Input: 9.0,118,95 | Prob: 0.19305570423603058 | Actual: 0

 Ground truth uploaded to: s3://placement-project-bkt/monitoring/ground-truth/2026/04/06/08/labels.jsonl


/tmp/ipykernel_19520/2098581674.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "inferenceTime": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
/tmp/ipykernel_19520/2098581674.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [12]:
import boto3

s3 = boto3.client("s3")

# Check captured file time
print("📥 CAPTURED DATA:")
response = s3.list_objects_v2(
    Bucket="placement-project-bkt",
    Prefix="monitoring/data-capture/"
)
for obj in sorted(response.get("Contents", []), key=lambda x: x["LastModified"], reverse=True)[:3]:
    print(f"🕐 {obj['LastModified']}  |  {obj['Key']}")

# Check ground truth time
print("\n📋 GROUND TRUTH:")
response = s3.list_objects_v2(
    Bucket="placement-project-bkt",
    Prefix="monitoring/ground-truth/"
)
for obj in sorted(response.get("Contents", []), key=lambda x: x["LastModified"], reverse=True)[:3]:
    print(f"🕐 {obj['LastModified']}  |  {obj['Key']}")

📥 CAPTURED DATA:
🕐 2026-04-06 08:40:24+00:00  |  monitoring/data-capture/Driftplacement-model-endpointt/AllTraffic/2026/04/06/08/39-16-390-76d6ee64-8f77-4f90-afaf-6e9d61950d4a.jsonl
🕐 2026-04-06 07:41:04+00:00  |  monitoring/data-capture/Driftplacement-model-endpointt/AllTraffic/2026/04/06/07/39-03-827-fbee0697-ef7e-4d4c-aacb-69c2290338ff.jsonl

📋 GROUND TRUTH:
🕐 2026-04-06 08:40:15+00:00  |  monitoring/ground-truth/2026/04/06/08/labels.jsonl
🕐 2026-04-06 07:44:22+00:00  |  monitoring/ground-truth/2026/04/06/07/labels.jsonl
